## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, ttest_ind
import warnings


warnings.filterwarnings('ignore')

## Load data

In [2]:
df = pd.read_csv("../data/insurance_cleaned.csv")

C:\Users\hp\AppData\Local\Temp\ipykernel_3868\1864962053.py:1: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/insurance_cleaned.csv")


## Step 1 — Create Core Risk Variables

### Claim Frequency

In [3]:
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

### Claim Severity

In [4]:
claims_only = df[df['TotalClaims'] > 0]

### SIGNIFICANCE LEVEL

In [5]:
alpha = 0.05

## HYPOTHESIS 1

H0: No risk differences across provinces
KPI: Claim Frequency
Test: Chi-Square

In [8]:
province_data = df[
    df['Province'].isin(['Gauteng', 'Western Cape'])
]

contingency_province = pd.crosstab(
    province_data['Province'],
    province_data['HasClaim']
)

chi2, p, dof, expected = chi2_contingency(contingency_province)

print("Province Claim Frequency Test")
print(contingency_province)
print(f"\nP-value: {p:.6f}")

if p < alpha:
    print("Reject H0: Significant risk difference exists across provinces.")
else:
    print("Fail to Reject H0: No significant risk difference detected.")

Province Claim Frequency Test
HasClaim           0     1
Province                  
Gauteng       392303  1322
Western Cape  170207   357

P-value: 0.000000
Reject H0: Significant risk difference exists across provinces.


## HYPOTHESIS 2

H0: No risk differences between zip codes
KPI: Claim Frequency
Test: Chi-Square

In [ ]:
# Select two high-volume postal codes
top_zipcodes = df['PostalCode'].value_counts().head(10)
print("Top Postal Codes:\n")
print(top_zipcodes)

Top Postal Codes:

PostalCode
2000    133258
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11562
2196     11048
470      10226
7100     10161
Name: count, dtype: int64


In [13]:
zip_a = top_zipcodes.index[0]
zip_b = top_zipcodes.index[1]

zipcode_data = df[
    df['PostalCode'].isin([zip_a, zip_b])
]

contingency_zip = pd.crosstab(
    zipcode_data['PostalCode'],
    zipcode_data['HasClaim']
)

chi2, p, dof, expected = chi2_contingency(contingency_zip)

print(f"\nZip Code Comparison: {zip_a} vs {zip_b}")
print("--------------------------------")
print(contingency_zip)
print(f"\nP-value: {p:.6f}")

if p < alpha:
    print("Reject H0: Significant risk difference exists between zip codes.")
else:
    print("Fail to Reject H0: No significant risk difference detected.")


Zip Code Comparison: 2000 vs 122
--------------------------------
HasClaim         0    1
PostalCode             
122          48961  210
2000        132772  486

P-value: 0.060832
Fail to Reject H0: No significant risk difference detected.


# HYPOTHESIS 3

H0: No significant margin difference between zip codes
KPI: Margin
Test: T-Test

In [14]:
group_a_margin = df[df['PostalCode'] == zip_a]['Margin']
group_b_margin = df[df['PostalCode'] == zip_b]['Margin']

t_stat, p = ttest_ind(
    group_a_margin,
    group_b_margin,
    equal_var=False,
    nan_policy='omit'
)

print(f"Margin Comparison: {zip_a} vs {zip_b}")
print("--------------------------------")
print(f"Average Margin ({zip_a}): {group_a_margin.mean():.2f}")
print(f"Average Margin ({zip_b}): {group_b_margin.mean():.2f}")
print(f"\nP-value: {p:.6f}")

if p < alpha:
    print("Reject H0: Significant margin difference exists.")
else:
    print("Fail to Reject H0: No significant margin difference detected.")

Margin Comparison: 2000 vs 122
--------------------------------
Average Margin (2000): -8.16
Average Margin (122): -22.86

P-value: 0.246238
Fail to Reject H0: No significant margin difference detected.


# HYPOTHESIS 4

H0: No significant risk difference between Women and Men
KPI: Claim Frequency
Test: Chi-Square

In [15]:
gender_data = df[
    df['Gender'].isin(['Male', 'Female'])
]

contingency_gender = pd.crosstab(
    gender_data['Gender'],
    gender_data['HasClaim']
)

chi2, p, dof, expected = chi2_contingency(contingency_gender)

print("Gender Claim Frequency Test")
print("--------------------------------")
print(contingency_gender)
print(f"\nP-value: {p:.6f}")

if p < alpha:
    print("Reject H0: Significant risk difference exists between genders.")
else:
    print("Fail to Reject H0: No significant risk difference detected.")

Gender Claim Frequency Test
--------------------------------
HasClaim      0   1
Gender             
Female     6741  14
Male      42723  94

P-value: 0.951464
Fail to Reject H0: No significant risk difference detected.


## Hypothesis Testing Results Summary

| Hypothesis                                               | KPI             | Statistical Test   | p-value | Decision          |
| -------------------------------------------------------- | --------------- | ------------------ | ------- | ----------------- |
| H₀: No risk differences across provinces                 | Claim Frequency | Chi-Square Test    | < 0.001 | Reject H₀         |
| H₀: No risk differences between zip codes                | Claim Frequency | Chi-Square Test    | 0.0608  | Fail to Reject H₀ |
| H₀: No significant margin difference between zip codes   | Margin          | Independent t-test | 0.2462  | Fail to Reject H₀ |
| H₀: No significant risk difference between Women and Men | Claim Frequency | Chi-Square Test    | 0.9515  | Fail to Reject H₀ |


# Business Interpretations and Recommendations

## 1. Risk Differences Across Provinces — Reject H₀

### Business Interpretation

We reject the null hypothesis for provinces because the p-value is below the 0.05 significance threshold. This indicates that claim frequency differs significantly between Gauteng and the Western Cape. Gauteng recorded a noticeably larger number of claims relative to policy volume, suggesting a higher underlying insurance risk profile in that province.

This finding implies that geographic location is an important driver of insurance risk and should be incorporated into ACIS’s segmentation and pricing strategy.

### Business Recommendations

* Introduce province-based risk pricing adjustments.
* Apply stricter underwriting controls in higher-risk provinces such as Gauteng.
* Prioritize low-risk provinces for customer acquisition campaigns and discounted premiums.
* Investigate additional regional risk drivers such as traffic density, theft frequency, and accident hotspots.

---

## 2. Risk Differences Between Zip Codes — Fail to Reject H₀

### Business Interpretation

We fail to reject the null hypothesis because the p-value exceeds 0.05. Although the selected postal codes exhibited some variation in claim frequency, the difference was not statistically significant.

This suggests that the observed differences may be due to random variation rather than meaningful geographic risk differences at the postal-code level for these specific locations.

### Business Recommendations

* Avoid implementing aggressive postal-code pricing differentiation based solely on the current evidence.
* Expand the analysis to include additional high-volume postal codes.
* Combine postal-code information with other variables such as vehicle type, income proxies, or urban density to improve segmentation accuracy.

---

## 3. Margin Differences Between Zip Codes — Fail to Reject H₀

### Business Interpretation

We fail to reject the null hypothesis because no statistically significant difference in profitability was detected between the selected postal codes. Although Postal Code 122 showed a lower average margin than Postal Code 2000, the variation was insufficient to establish a reliable profitability distinction.

This indicates that postal code alone may not strongly influence portfolio profitability.

### Business Recommendations

* Continue using broader segmentation variables in profitability analysis rather than relying solely on zip codes.
* Investigate additional profitability drivers such as vehicle value, claim severity, and policy type.
* Consider multivariate modeling approaches for more precise profit segmentation.

---

## 4. Risk Differences Between Women and Men — Fail to Reject H₀

### Business Interpretation

We fail to reject the null hypothesis because the p-value indicates no statistically significant difference in claim frequency between male and female policyholders within the analyzed sample.

This suggests that gender alone may not be a meaningful predictor of insurance risk in the current portfolio.

### Business Recommendations

* Avoid using gender as a primary segmentation or pricing variable.
* Focus instead on behavioral and vehicle-related risk indicators with stronger predictive power.
* Continue monitoring demographic trends as the customer portfolio expands.